In [1]:
%load_ext autoreload
%autoreload 2

#%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from datetime import datetime, timedelta
from ipywidgets import interact
from tqdm.notebook import tqdm
import plotly.express as px
from scipy import stats
import warnings
pd.options.mode.chained_assignment = None 


import sys
sys.path.append('../')
from eventbee import chronogram

# Load full dataframe of events
full_df = pd.read_parquet('one_week_analysis_800__tracks__with_labelbee_videoid.parquet')
full_df.shape

# Select tag ids that have 5 events or more
id_counts = full_df['track_tagid'].dropna().astype(int).value_counts().rename('counts')
tids = id_counts[id_counts>=5].index
tids = np.sort(tids).astype(int)

# Custom tids
#tids = np.array([1609,1621],dtype=int)

# Restrict data to selected tag ids
vdf = full_df[full_df.track_tagid.isin(tids)].copy()
vdf['track_tagid'] = vdf['track_tagid'].astype(int) # Convert to int after removing the NaNs
vdf.shape

ModuleNotFoundError: No module named 'eventbee'

In [ ]:
def beeClean2(bee):

    bee['track_event'] = bee['track_event'].apply(lambda x: 'None' if x == None else x)
    init = bee['datetime'].iloc[0]
    bee['time'] = bee['datetime'].apply(lambda x: (x - init).total_seconds())
    id = bee['track_tagid'].iloc[0]

    #classify noise
    for track in range(len(bee)):
        if bee['track_shape'].iloc[track] == 'noise' or bee['track_shape'].iloc[track] == 'ramp_ramp' or bee['track_shape'].iloc[track] == None:  
            start = bee['track_starty'].iloc[track]
            end = bee['track_endy'].iloc[track]
            if start <= 300:
                init = 'inside'
            elif start <= 800:
                init = 'ramp'
            else:
                init = 'outside'
            if end <= 300:
                ed = 'inside'
            elif end <= 800:
                ed = 'ramp'
            else:
                ed = 'outside'
    
            bee['track_shape'].iloc[track] = f'{init}_{ed}'
            
            if bee['track_shape'].iloc[track] == "ramp_ramp":
                if end > start:
                    bee['track_shape'].iloc[track] = "ramp_outside"
                else:
                    bee['track_shape'].iloc[track] = "ramp_inside"
                    
            if bee['track_shape'].iloc[track] == "inside_inside":
                if end > start:
                    bee['track_shape'].iloc[track] = "inside_ramp"
                else:
                    bee['track_shape'].iloc[track] = "ramp_inside"
    
            if bee['track_shape'].iloc[track] == "outside_outside":
                if end > start:
                    bee['track_shape'].iloc[track] = "ramp_outside"
                else:
                    bee['track_shape'].iloc[track] = "outside_ramp"

    bee['track_shape'] = bee['track_shape'].apply(lambda x: 'ramp_inside' if x == 'ramp-inside' else x)

    #new dataframe dict
    new_event = []
    datetime = []
        
    t = 60 #seconds threshold
    
    for i in range(0,len(bee)-1):
        shape = bee['track_shape'].iloc[i]
        time = bee['datetime'].iloc[i]
        next_t = bee['datetime'].iloc[i+1]

        entering = ["outside_inside", "inside_inside", "inside", "ramp_inside", "outside_ramp"]
        leaving = ["inside_outside", "inside_out", "outside", "outside_outside", "ramp_outside", "inside_ramp"]
        if (next_t - time).total_seconds() > t:
            #classify
            if shape in entering:
                new_event.append('entering')
            else:
                new_event.append('exiting')
            datetime.append(time)

    tagID = [id] * len(new_event)
    df = pd.DataFrame.from_dict({'tagID': tagID, 'datetime':datetime, 'event':new_event})
    return df

vdf_new = vdf.drop(['video_name','labelbee_videoid','video_seq','video_filename','pollen','track_startx','track_endx','track_starta','track_enda','track_startframe','track_endframe','track_hastag','entering','leaving','entering_leaving','walking','track_length','track_id'], axis=1)

bees = []

beeIDs = vdf['track_tagid'].unique()
for bee in beeIDs:

    b = vdf_new[vdf_new['track_tagid'] == bee].copy().reset_index()
    events = beeClean2(b)
    bees.append(events)
    
    
new = pd.concat(bees, axis = 0) 
new['tagID'] = new['tagID'].apply(lambda x: int(x))
new.to_csv("cleandata.csv", index=False)
new